# Clasificación CIFAR-10 con PyTorch


## Introducción

El dataset CIFAR-10 es una colección de 60,000 imágenes en color de 32x32 píxeles, que contiene 10 clases de objetos.

![cifar10](https://pytorch.org/tutorials/_static/img/cifar10.png)

## Configuración del Entorno

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import os

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

# Determinar el dispositivo (GPU si está disponible, de lo contrario CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True # Elegir el mejor algoritmo según el hardware
    torch.cuda.manual_seed_all(42) # Inicializar la semilla para la GPU
else:
    device = torch.device("cpu")
    
print(f"\nUsando dispositivo: {device}")


num_workers = os.cpu_count() // 2 # Número de workers. La mitad de los núcleos disponibles (heurística común)

## Carga y Preparación de Datos

Usaremos el [tutorial oficial de PyTorch para la clasificación CIFAR-10](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) como modelo de referencia (*baseline*) y buscaremos mejorar su rendimiento.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalización 'Perezosa/Simétrica' con 0.5 para todos los canales
])

batch_size = 4

fulltrainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# Separación en conjuntos de entrenamiento y validación
train_size = int(0.8 * len(fulltrainset))
valid_size = len(fulltrainset) - train_size
trainset, validset = torch.utils.data.random_split(fulltrainset, [train_size, valid_size])

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
validloader = torch.utils.data.DataLoader(validset, batch_size=batch_size, shuffle=False, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

fulltrainset.classes # Las clases ya están definidas en el dataset

## Definición de las Funciones de Entrenamiento y Validación

Existen diferentes metodologías de entrenamiento y evaluación.
- Enfoque 1: Entrenar completamente, luego evaluar (funciones separadas)
    - Simplicidad: Código más simple y fácil de entender.
    - Velocidad: Más rápido si solo te interesa el resultado final.
    - Menos sobrecarga: No interrumpe el proceso de entrenamiento.
    - Adecuado para grandes datasets: Cuando la evaluación es costosa.
- Enfoque 2: Evaluar tras cada época (función integrada)
    - Monitorización del progreso: Puedes ver cómo mejora el modelo durante el entrenamiento.
    - Detección temprana de problemas: Identifica *overfitting/underfitting* a tiempo.
    - *Early stopping*: Puedes detener el entrenamiento cuando deja de mejorar.
    - Guardar el mejor modelo: Guarda el modelo con el mejor rendimiento en validación.
    
En este caso usamos el enfoque 2, que permite comparar diferentes arquitecturas observando el progreso a lo largo de las épocas. Así, visualizamos no solo los resultados finales sino también la curva de aprendizaje.

Se definen funciones de entrenamiento y evaluación para que puedan reutilizarse al entrenar diferentes arquitecturas.

In [ ]:
from utils.training_functions import train_model, evaluate_model, train_final_model
from utils.plot_functions import plot_metrics, plot_class_accuracy

## *Baseline*

In [ ]:
# Inicializar el modelo baseline
from models.baseline import Net
base_model = Net()

# Definir el criterio de pérdida y el optimizador
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(base_model.parameters(), lr=0.001, momentum=0.9)

# Entrenar el modelo baseline
base_metrics = train_model(
    base_model, 
    trainset,
    validset,
    batch_size,
    criterion,
    optimizer,
    epochs=15,
    device=device,
    num_workers=num_workers
)

plot_metrics([base_metrics], ['Modelo Base'])

Podemos observar *overfitting* ya que la precisión de entrenamiento aumenta mucho más que la precisión de validación a partir de aproximadamente la época 10.

## Mejora

Para evitar el *overfitting*, podemos usar regularización, Dropout, etc.

In [ ]:
from models.mejorado import ImprovedCNN
model2 = ImprovedCNN()

# Definir el criterio de pérdida y el optimizador
criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model2.parameters(), lr=0.001, momentum=0.9)
optimizer = torch.optim.AdamW(model2.parameters(), lr=0.001, weight_decay=1e-4)

# Entrenar el modelo
metrics2 = train_model(
    model2, 
    trainset,
    validset,
    32,
    criterion,
    optimizer,
    epochs=15,
    device=device,
    num_workers=num_workers
)

### Visualización de métricas
plot_metrics([base_metrics, metrics2], ['Modelo Base', "Modelo2"])

## Evaluación en Test y Almacenamiento del Modelo Final

Habiendo elegido la nueva arquitectura propuesta como modelo final, lo reentrenamos con todo el conjunto de entrenamiento.

In [ ]:
model2 = ImprovedCNN() # Reinicializar el modelo (aunque podrían existir otras estrategias)
optimizer = torch.optim.AdamW(model2.parameters(), lr=0.001, weight_decay=1e-4)

final_metrics = train_final_model(
    model2,
    fulltrainset,
    batch_size,
    criterion,
    optimizer,
    epochs=10, # A partir de aquí vimos que comenzaba a producirse overfitting
    device=device,
    num_workers=num_workers,
)

In [ ]:
# Evaluar el modelo final en el conjunto de prueba
test_results = evaluate_model(model2, testloader, criterion, device) # Capturar los resultados completos
test_loss = test_results['loss']
test_acc = test_results['accuracy']
print(f"Pérdida en Test: {test_loss:.4f}, Precisión en Test: {test_acc:.2f}%")

# Ahora graficar la precisión por clase usando los resultados capturados
plot_class_accuracy(
    results_list=[test_results],  # Pasar el diccionario de resultados en una lista
    model_names=['Modelo Final'],  # Proporcionar un nombre para el modelo
    class_names=testset.classes
)

Una vez elegido y entrenado el modelo, podemos almacenarlo para su despliegue.

In [ ]:
PATH = './cifar_net.pth'
torch.save(model2.state_dict(), PATH)

## Despliegue del Modelo en Hugging Face Hub

Por último, podemos automatizar la subida del modelo entrenado al Hugging Face Model Hub. Esto permite que nuestra aplicación FastAPI (ejecutándose en Hugging Face Spaces) descargue dinámicamente los pesos del modelo más reciente.

> **Requisito de Autenticación**: 
> Para subir modelos a Hugging Face, necesitas un Token de Acceso con permisos de **ESCRITURA**. 
> Tienes dos opciones seguras para proporcionar este token:
> 1. **Archivo `.env`**: Crea un archivo `.env` en la carpeta base y añade `HF_TOKEN=tu_token_aqui`. Luego reinicia tu entorno, o simplemente expórtalo en tu shell (`export HF_TOKEN=...`).
> 2. **Login Interactivo**: Usa `huggingface_hub.notebook_login()` para pegar tu token de forma interactiva.

In [ ]:
import os
from huggingface_hub import HfApi, notebook_login

# Buscar HF_TOKEN en las variables de entorno
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN no encontrado en las variables de entorno. Por favor, inicia sesión de forma interactiva:")
    notebook_login()
else:
    print("¡HF_TOKEN encontrado en las variables de entorno!")

In [ ]:
api = HfApi()

repo_id = "avidaldo/cifar-10-fastapi-model" # Repositorio de modelo destino en Hugging Face

# Crear el repositorio del modelo si no existe
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Subir el archivo del modelo
print(f"Subiendo ./cifar_net.pth a {repo_id}...")
api.upload_file(
    path_or_fileobj='./cifar_net.pth',
    path_in_repo="cifar_net.pth",
    repo_id=repo_id,
    repo_type="model"
)

print(f"Modelo subido correctamente a https://huggingface.co/{repo_id}")